# Incident Triage Notebook

**End-to-end workflow:**
1. Install dependencies
2. Configuration
3. Phase A — Training & Evaluation
4. Save / Load Artifacts
5. Phase B — Interactive Triage (prompts for ticket number)


## 1  Install Dependencies


In [ ]:
# Run once in a fresh environment
!pip -q install transformers datasets torch scikit-learn lightgbm faiss-cpu sentence-transformers joblib tqdm


## 2  Configuration


In [ ]:
import os, json, random, logging, warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)
log = logging.getLogger('triage')

# ── Artifact output directory ──
ARTIFACT_DIR = Path('artifacts')
ARTIFACT_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ── Model identifiers ──
DEBERTA_MODEL_NAME  = 'microsoft/deberta-v3-base'
EMOTION_MODEL_NAME  = 'j-hartmann/emotion-english-distilroberta-base'
MINILM_MODEL_NAME   = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
BGE_MODEL_NAME      = 'BAAI/bge-base-en-v1.5'

# ── Tenant / process-server maps ──
TENANTS = ['acme', 'globex', 'initech', 'umbrella']
PROCESS_SERVERS = ['processserver-0', 'processserver-1', 'processserver-2']

CATEGORIES = ['Authentication', 'Performance', 'DataLoss', 'Connectivity', 'Other']
PRIORITIES = ['P1', 'P2', 'P3', 'P4', 'P5']

print('Configuration complete.')


## 3  Synthetic Training Data


In [ ]:
def make_synthetic_tickets(n: int = 400) -> pd.DataFrame:
    """Generate synthetic labeled tickets for demo training."""
    templates = [
        ('Authentication', 'P1', 'Users cannot log in to {tenant} portal — password reset broken'),
        ('Authentication', 'P2', 'SSO token expiry too short on {tenant} causing repeated re-authentication'),
        ('Performance',    'P2', 'Dashboard latency exceeds 10 s for {tenant} after deploy on {ps}'),
        ('Performance',    'P3', 'Report generation slow on {tenant} {ps}'),
        ('DataLoss',       'P1', 'Critical data missing from {tenant} export — rows dropped silently'),
        ('DataLoss',       'P2', 'Partial write failure on {tenant} {ps} under heavy load'),
        ('Connectivity',   'P2', '{tenant} cannot reach {ps} intermittently'),
        ('Connectivity',   'P3', 'WebSocket disconnections for {tenant} clients on {ps}'),
        ('Other',          'P3', 'Scheduled job not triggered for {tenant} on {ps}'),
        ('Other',          'P4', 'Minor UI glitch on {tenant} admin page'),
    ]
    rows = []
    for i in range(n):
        cat, pri, tmpl = templates[i % len(templates)]
        text = tmpl.format(
            tenant=random.choice(TENANTS),
            ps=random.choice(PROCESS_SERVERS)
        )
        risk = 1 if pri in ('P1', 'P2') else 0
        rows.append({'id': f'TICKET-{1000+i}', 'text': text, 'category': cat,
                     'priority': pri, 'risk': risk})
    return pd.DataFrame(rows)


df = make_synthetic_tickets(400)
print(df.shape)
df.head()


## 4  Train / Validation / Test Split


In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.3, random_state=RANDOM_SEED, stratify=df['category'])
val_df,   test_df = train_test_split(temp_df, test_size=0.5, random_state=RANDOM_SEED)

print(f'Train={len(train_df)}  Val={len(val_df)}  Test={len(test_df)}')


## 5A  Ticket Classification — DeBERTa-v3-base


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import f1_score, accuracy_score

cat2id = {c: i for i, c in enumerate(CATEGORIES)}
id2cat = {v: k for k, v in cat2id.items()}

def encode_category(row):
    return {'label': cat2id[row['category']]}

tokenizer_cls = AutoTokenizer.from_pretrained(DEBERTA_MODEL_NAME)

def tokenize_cls(batch):
    return tokenizer_cls(batch['text'], truncation=True, padding='max_length', max_length=128)

def to_hf_dataset(data_df):
    ds = Dataset.from_pandas(data_df[['text', 'category']].reset_index(drop=True))
    ds = ds.map(encode_category)
    ds = ds.map(tokenize_cls, batched=True)
    ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
    return ds

train_ds = to_hf_dataset(train_df)
val_ds   = to_hf_dataset(val_df)
test_ds  = to_hf_dataset(test_df)

model_cls = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_MODEL_NAME, num_labels=len(CATEGORIES), ignore_mismatched_sizes=True
)

def compute_metrics_cls(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro')
    }

training_args_cls = TrainingArguments(
    output_dir=str(ARTIFACT_DIR / 'deberta_cls'),
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_steps=20,
    seed=RANDOM_SEED,
    report_to='none',
)

trainer_cls = Trainer(
    model=model_cls,
    args=training_args_cls,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics_cls,
)

trainer_cls.train()
cls_test_metrics = trainer_cls.evaluate(test_ds)
print('DeBERTa test metrics:', cls_test_metrics)


## 5B  Fallback Classifier — TF-IDF + Logistic Regression


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import joblib

fallback_clf = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ('lr',    LogisticRegression(max_iter=500, random_state=RANDOM_SEED)),
])
fallback_clf.fit(train_df['text'], train_df['category'])
fb_preds = fallback_clf.predict(test_df['text'])
print('Fallback classifier report:')
print(classification_report(test_df['category'], fb_preds))

joblib.dump(fallback_clf, ARTIFACT_DIR / 'fallback_clf.joblib')
print('Fallback classifier saved.')


## 6  Priority Prediction — LightGBM Multiclass


In [ ]:
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer as TfIdf

pri_enc = LabelEncoder()
pri_enc.fit(PRIORITIES)

tfidf_pri = TfIdf(max_features=3000)
X_train_pri = tfidf_pri.fit_transform(train_df['text'])
X_val_pri   = tfidf_pri.transform(val_df['text'])
X_test_pri  = tfidf_pri.transform(test_df['text'])

y_train_pri = pri_enc.transform(train_df['priority'])
y_val_pri   = pri_enc.transform(val_df['priority'])
y_test_pri  = pri_enc.transform(test_df['priority'])

lgb_pri = lgb.LGBMClassifier(
    objective='multiclass', num_class=len(PRIORITIES),
    n_estimators=200, learning_rate=0.05, random_state=RANDOM_SEED, verbose=-1
)
lgb_pri.fit(X_train_pri, y_train_pri,
            eval_set=[(X_val_pri, y_val_pri)])
pri_preds = lgb_pri.predict(X_test_pri)
print('Priority F1 (macro):', f1_score(y_test_pri, pri_preds, average='macro').round(4))

joblib.dump({'model': lgb_pri, 'tfidf': tfidf_pri, 'encoder': pri_enc},
            ARTIFACT_DIR / 'priority_model.joblib')
print('Priority model saved.')


## 7  Emotion Detection — DistilRoBERTa


In [ ]:
from transformers import pipeline as hf_pipeline

emotion_pipe = hf_pipeline('text-classification', model=EMOTION_MODEL_NAME, top_k=1, device=-1)

sample_texts = [
    'Users cannot log in — this is critical!',
    'Minor UI glitch on admin page',
]
for t in sample_texts:
    result = emotion_pipe(t)
    print(f'{t!r}  =>  {result[0][0]["label"]} ({result[0][0]["score"]:.3f})')

print('Emotion model loaded and verified.')


## 8  Log Relevance Reranker — MiniLM


In [ ]:
from sentence_transformers import CrossEncoder

minilm = CrossEncoder(MINILM_MODEL_NAME, max_length=256)

def score_log_relevance(query: str, log_lines: list) -> list:
    pairs = [(query, line) for line in log_lines]
    scores = minilm.predict(pairs)
    ranked = sorted(zip(log_lines, scores.tolist()), key=lambda x: x[1], reverse=True)
    return ranked

sample_query = 'authentication failure SSO'
sample_logs  = [
    'ERROR 2025-01-10 auth-service: SSO token validation failed for user admin',
    'INFO  2025-01-10 payment-service: transaction completed',
    'WARN  2025-01-10 auth-service: retry limit reached',
]
ranked_logs = score_log_relevance(sample_query, sample_logs)
for line, score in ranked_logs:
    print(f'{score:6.3f}  {line}')


## 9  Semantic Retrieval — BGE-base-en-v1.5 + FAISS


In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

bge_model = SentenceTransformer(BGE_MODEL_NAME)

# Build a small corpus from training data
corpus_texts = train_df['text'].tolist()
corpus_ids   = train_df['id'].tolist()

corpus_embeddings = bge_model.encode(corpus_texts, normalize_embeddings=True, show_progress_bar=True)
dim = corpus_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(corpus_embeddings.astype('float32'))
faiss.write_index(faiss_index, str(ARTIFACT_DIR / 'bge_faiss.index'))

import pickle
with open(ARTIFACT_DIR / 'corpus_ids.pkl', 'wb') as f:
    pickle.dump(corpus_ids, f)

def retrieve(query: str, k: int = 5) -> list:
    qvec = bge_model.encode([query], normalize_embeddings=True).astype('float32')
    scores, idxs = faiss_index.search(qvec, k)
    return [(corpus_ids[i], corpus_texts[i], float(scores[0][j])) for j, i in enumerate(idxs[0])]

hits = retrieve('SSO authentication timeout', k=3)
for tid, txt, sc in hits:
    print(f'{sc:.4f}  [{tid}]  {txt}')

print('BGE + FAISS index built and saved.')


## 10  Incident Risk Prediction — LightGBM Binary


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

tfidf_risk = TfIdf(max_features=3000)
X_train_risk = tfidf_risk.fit_transform(train_df['text'])
X_val_risk   = tfidf_risk.transform(val_df['text'])
X_test_risk  = tfidf_risk.transform(test_df['text'])

lgb_risk = lgb.LGBMClassifier(
    objective='binary', n_estimators=200, learning_rate=0.05,
    random_state=RANDOM_SEED, verbose=-1
)
lgb_risk.fit(X_train_risk, train_df['risk'],
             eval_set=[(X_val_risk, val_df['risk'])])

risk_proba = lgb_risk.predict_proba(X_test_risk)[:, 1]
print('Risk AUC:  ', roc_auc_score(test_df['risk'], risk_proba).round(4))
print('Risk AP:   ', average_precision_score(test_df['risk'], risk_proba).round(4))

joblib.dump({'model': lgb_risk, 'tfidf': tfidf_risk},
            ARTIFACT_DIR / 'risk_model.joblib')
print('Risk model saved.')


## 11  Metrics Summary


In [ ]:
metrics_summary = {
    'deberta_cls':  cls_test_metrics,
    'fallback_f1':  float(f1_score(test_df['category'], fb_preds, average='macro')),
    'priority_f1':  float(f1_score(y_test_pri, pri_preds, average='macro')),
    'risk_auc':     float(roc_auc_score(test_df['risk'], risk_proba)),
}
with open(ARTIFACT_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics_summary, f, indent=2)

pd.DataFrame([metrics_summary])


## 12  Save / Load Artifacts


In [ ]:
def load_artifacts():
    """Load all trained artifacts for inference."""
    fallback = joblib.load(ARTIFACT_DIR / 'fallback_clf.joblib')
    pri_bundle = joblib.load(ARTIFACT_DIR / 'priority_model.joblib')
    risk_bundle = joblib.load(ARTIFACT_DIR / 'risk_model.joblib')

    idx = faiss.read_index(str(ARTIFACT_DIR / 'bge_faiss.index'))
    with open(ARTIFACT_DIR / 'corpus_ids.pkl', 'rb') as f:
        c_ids = pickle.load(f)

    return {
        'fallback_clf':  fallback,
        'pri_model':     pri_bundle['model'],
        'pri_tfidf':     pri_bundle['tfidf'],
        'pri_encoder':   pri_bundle['encoder'],
        'risk_model':    risk_bundle['model'],
        'risk_tfidf':    risk_bundle['tfidf'],
        'bge_index':     idx,
        'corpus_ids':    c_ids,
        'corpus_texts':  corpus_texts,
        'bge_model':     bge_model,
        'minilm':        minilm,
        'emotion_pipe':  emotion_pipe,
        'deberta_model': model_cls,
        'deberta_tok':   tokenizer_cls,
        'id2cat':        id2cat,
    }

artifacts = load_artifacts()
print('All artifacts loaded.')


---

# Phase B — Interactive Incident Triage

Run Phase A first (or reload a saved run).  
Then execute the cells below to triage a specific ticket.


## 13  Tenant & Process-Server Discovery


In [ ]:
import re

def resolve_tenant(text: str) -> str:
    """Extract tenant from ticket text; fall back to 'unknown'."""
    text_lower = text.lower()
    for t in TENANTS:
        if t in text_lower:
            return t
    return 'unknown'


def resolve_process_server(text: str, tenant: str) -> str:
    """Extract process server from ticket text; assign deterministically."""
    m = re.search(r'processserver-(\d+)', text, re.IGNORECASE)
    if m:
        return f'processserver-{m.group(1)}'
    # Deterministic fallback: hash tenant to a server index
    idx = abs(hash(tenant)) % len(PROCESS_SERVERS)
    return PROCESS_SERVERS[idx]


print('Tenant/process-server resolver ready.')


## 14  Interactive Ticket Input


In [ ]:
# ─────────────────────────────────────────────────────────────────
# INTERACTIVE PROMPT — enter your Jira ticket number when prompted
# ─────────────────────────────────────────────────────────────────
ticket_number = input('Enter Jira ticket number (e.g. TICKET-1042): ').strip()

# Look up the ticket in the dataframe (demo: use synthetic data)
match = df[df['id'] == ticket_number]
if not match.empty:
    ticket_text     = match.iloc[0]['text']
    ticket_actual   = match.iloc[0].to_dict()
else:
    # Treat the input itself as the ticket description
    ticket_text   = ticket_number
    ticket_actual = {'id': ticket_number, 'text': ticket_text}

print(f'Ticket  : {ticket_actual.get("id", ticket_number)}')
print(f'Text    : {ticket_text}')


## 15  Inference Pipeline


In [ ]:
def classify_ticket(text: str, art: dict) -> dict:
    """Classify ticket with DeBERTa; fall back to TF-IDF LR if confidence low."""
    tok  = art['deberta_tok']
    mdl  = art['deberta_model']
    mdl.eval()
    enc = tok(text, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        logits = mdl(**enc).logits
    probs  = torch.softmax(logits, dim=-1).squeeze().tolist()
    top_id = int(np.argmax(probs))
    confidence = probs[top_id]
    if confidence < 0.5:
        # Use fallback
        fb_label = art['fallback_clf'].predict([text])[0]
        return {'category': fb_label, 'confidence': confidence, 'source': 'fallback'}
    return {'category': art['id2cat'][top_id], 'confidence': confidence, 'source': 'deberta'}


def predict_priority(text: str, art: dict) -> str:
    xvec = art['pri_tfidf'].transform([text])
    return art['pri_encoder'].inverse_transform(art['pri_model'].predict(xvec))[0]


def predict_risk(text: str, art: dict) -> float:
    xvec = art['risk_tfidf'].transform([text])
    return float(art['risk_model'].predict_proba(xvec)[0, 1])


def detect_emotion(text: str, art: dict) -> dict:
    result = art['emotion_pipe'](text)
    top = result[0][0]
    return {'emotion': top['label'], 'score': round(top['score'], 4)}


def semantic_search(text: str, art: dict, k: int = 3) -> list:
    qvec = art['bge_model'].encode([text], normalize_embeddings=True).astype('float32')
    scores, idxs = art['bge_index'].search(qvec, k)
    return [
        {'id': art['corpus_ids'][i], 'text': art['corpus_texts'][i], 'score': float(scores[0][j])}
        for j, i in enumerate(idxs[0])
    ]


def find_relevant_logs(text: str, art: dict) -> list:
    """Simulate dynamic log discovery and MiniLM reranking."""
    candidate_logs = [
        f'ERROR auth-service: token validation failed for session related to {text[:30]}',
        'INFO  payment-service: transaction completed successfully',
        f'WARN  processserver: high memory usage detected near {text[:30]}',
        'DEBUG scheduler: heartbeat ok',
        f'ERROR db-service: query timeout for tenant context {text[:20]}',
    ]
    ranked = score_log_relevance(text, candidate_logs)
    return [{'log': l, 'score': round(s, 4)} for l, s in ranked[:3]]


print('Inference functions defined.')


## 16  Run Triage Inference


In [ ]:
cls_result   = classify_ticket(ticket_text, artifacts)
priority     = predict_priority(ticket_text, artifacts)
risk_score   = predict_risk(ticket_text, artifacts)
emotion      = detect_emotion(ticket_text, artifacts)
similar      = semantic_search(ticket_text, artifacts, k=3)
relevant_logs = find_relevant_logs(ticket_text, artifacts)
tenant       = resolve_tenant(ticket_text)
proc_server  = resolve_process_server(ticket_text, tenant)

triage_result = {
    'ticket_id':       ticket_actual.get('id', ticket_number),
    'text':            ticket_text,
    'category':        cls_result,
    'priority':        priority,
    'risk_score':      round(risk_score, 4),
    'is_high_risk':    risk_score >= 0.5,
    'emotion':         emotion,
    'tenant':          tenant,
    'process_server':  proc_server,
    'similar_tickets': similar,
    'relevant_logs':   relevant_logs,
}

print(json.dumps(triage_result, indent=2))


## 17  RCA Generation


In [ ]:
def generate_rca(result: dict) -> str:
    cat  = result['category']['category']
    pri  = result['priority']
    risk = result['risk_score']
    emo  = result['emotion']['emotion']
    tenant = result['tenant']
    ps     = result['process_server']

    lines = [
        f'## Root Cause Analysis — {result["ticket_id"]}',
        '',
        f'**Category**      : {cat}',
        f'**Priority**      : {pri}',
        f'**Risk Score**    : {risk} ({"HIGH" if result["is_high_risk"] else "LOW"})',
        f'**Tenant**        : {tenant}',
        f'**Process Server**: {ps}',
        f'**User Sentiment**: {emo}',
        '',
        '### Evidence',
    ]
    for lg in result['relevant_logs']:
        lines.append(f'- [{lg["score"]}] {lg["log"]}')
    lines.append('')
    lines.append('### Similar Historical Tickets')
    for st in result['similar_tickets']:
        lines.append(f'- {st["id"]} (score={st["score"]:.4f}): {st["text"]}')
    lines.append('')
    lines.append('### Recommended Actions')
    if cat == 'Authentication':
        lines.append('- Review SSO/auth service logs for token validation errors.')
        lines.append('- Check identity provider connectivity.')
    elif cat == 'Performance':
        lines.append('- Profile slow queries and check resource utilisation on ' + ps + '.')
    elif cat == 'DataLoss':
        lines.append('- Audit write transactions and enable idempotency checks.')
    elif cat == 'Connectivity':
        lines.append('- Inspect network policies and firewall rules between tenant and ' + ps + '.')
    else:
        lines.append('- Review application logs and escalate to on-call engineer if unresolved.')
    if result['is_high_risk']:
        lines.append('- **Escalate immediately** — risk score above threshold.')
    if emo in ('anger', 'disgust', 'fear'):
        lines.append('- **Customer sentiment is negative** — prioritise communication update.')
    return '\n'.join(lines)


rca_text = generate_rca(triage_result)
print(rca_text)


## 18  Notifications


In [ ]:
def send_notifications(result: dict, rca: str) -> None:
    """Log notification actions (replace with real email/Jira client in production)."""
    if result['is_high_risk']:
        log.warning('[NOTIFY] High-risk incident — on-call page sent for %s', result['ticket_id'])
    if result['emotion']['emotion'] in ('anger', 'disgust', 'fear'):
        log.info('[NOTIFY] Negative sentiment detected — customer update queued for %s', result['ticket_id'])
    log.info('[NOTIFY] RCA summary posted to Jira for %s', result['ticket_id'])
    print('Notifications dispatched (see log above).')


send_notifications(triage_result, rca_text)


## 19  Final Summary


In [ ]:
print('=' * 60)
print(f'Ticket       : {triage_result["ticket_id"]}')
print(f'Category     : {triage_result["category"]["category"]}  '
      f'(conf={triage_result["category"]["confidence"]:.3f}, '
      f'src={triage_result["category"]["source"]})')
print(f'Priority     : {triage_result["priority"]}')
print(f'Risk         : {triage_result["risk_score"]} '
      f'({"HIGH" if triage_result["is_high_risk"] else "LOW"})')
print(f'Emotion      : {triage_result["emotion"]["emotion"]} '
      f'({triage_result["emotion"]["score"]})')
print(f'Tenant       : {triage_result["tenant"]}')
print(f'ProcServer   : {triage_result["process_server"]}')
print('=' * 60)
